# Semana 8: análise exploratória E5, o R0 pelo pipeline da semana 6

E5 de `prereg/plano_exploratorio.md`. **É exploratório** e não muda a pré-condição 2 registrada pela rodada confirmatória.

Pergunta: a pré-condição 2 falhou no AUROC_e da Bolívia (R0 0,9432, máximo dos seeds do R3 0,9398). O R0 da semana 1 foi inferido com outro código (`LightningInferenceModel` do TerraTorch, configuração do próprio checkpoint), e o R1 a R3 com o código da semana 6. Este notebook roda o checkpoint publicado do R0 (commit `918b9f1`, conferido pelo SHA-256) **com o mesmo código de leitura, pré-processamento e inferência em 32 bits da semana 6** na validação, no teste da fonte e na Bolívia. Depois `analysis/exploratoria/e5_r0.py` calcula τ na validação e AUROC_e, erro não sinalizado, taxa de erro e mIoU, ao lado dos valores da semana 1 e da faixa do R3.

| Célula | O que faz |
|---|---|
| 1 | Instala TerraTorch 1.2.11 (mesmas versões da semana 6) |
| 2 | Monta o Drive, confere pastas e clona o repositório |
| 3 | Confere o backbone pelo hash (a montagem do modelo usa o mesmo caminho da semana 6) |
| 4 | Confere os pesos e o `config.yaml` do R0 pelo hash e compara a receita do `config.yaml` com a da semana 6 |
| 5 | Baixa as imagens e rótulos do Sen1Floods11 e confere pelos hashes da semana 6 |
| 6 | Funções de leitura (cópia da célula 5 da semana 6) |
| 7 | Lê val, teste e Bolívia e confere os rótulos contra os `labels_<conjunto>.npz` da semana 6 |
| 8 | Definições do modelo e da inferência (cópia da célula 7 da semana 6) |
| 9 | Carrega o R0 nessa montagem, com todos os tensores conferidos |
| 10 | Inferência em 32 bits na val, no teste e na Bolívia (alguns minutos) |
| 11 | Calcula as métricas e copia a saída para `resultados/semana8/e5` |

Abra o Colab logada na conta do mestrado, **com GPU T4**. Se a célula 1 pedir para reiniciar a sessão, reinicie e siga da célula 2. Se alguma célula der erro, pare e me mande a saída inteira.

Depois da célula 11, me mande de `resultados/semana8/e5`: `e5_r0.csv`, `e5.log` e `rodada.json`.

In [ ]:
!pip install -q "terratorch[peft]==1.2.11" "torchgeo==0.7.1" "peft==0.21.2"
!pip uninstall -y -q torchao

from importlib.metadata import version
for pacote in ('terratorch', 'torchgeo', 'peft', 'lightning', 'torch', 'numpy'):
    print(f'{pacote:11s} {version(pacote)}')
assert version('terratorch') == '1.2.11', 'Versão do TerraTorch diferente da fixada.'

import sys
if 'numpy' in sys.modules and sys.modules['numpy'].__version__ != version('numpy'):
    print(f"\nREINICIE A SESSÃO: o NumPy carregado é o {sys.modules['numpy'].__version__} e o instalado é o {version('numpy')}.")
    print('Menu Ambiente de execução > Reiniciar sessão (não precisa excluir o ambiente) e rode a partir da célula 2.')
else:
    print('\nNumPy carregado e instalado coincidem. Pode seguir para a célula 2.')

In [ ]:
import os, re, json, time, hashlib, datetime, shutil, glob, gc, csv, subprocess, sys
import numpy as np

from google.colab import drive
drive.mount('/content/drive')
MEUDRIVE = '/content/drive/MyDrive'
RAIZ = f'{MEUDRIVE}/Mestrado'
FIXADO = f'{RAIZ}/modelo_fixado'
assert os.path.isdir(FIXADO), 'Pasta Mestrado/modelo_fixado não encontrada. Confira se o Colab está na conta do mestrado.'

MARGENS = f'{RAIZ}/resultados/semana6/margens'
CONFIRMATORIA = f'{RAIZ}/resultados/semana6/analise'
HASHES_S1F11 = f'{RAIZ}/resultados/semana6/entradas_sen1floods11_sha256.csv'
SAIDA_DRIVE = f'{RAIZ}/resultados/semana8/e5'
for caminho in (f'{MARGENS}/labels_val.npz', f'{CONFIRMATORIA}/resultados.json', HASHES_S1F11):
    assert os.path.isfile(caminho), f'Arquivo não encontrado: {caminho}'
assert not (os.path.isdir(SAIDA_DRIVE) and os.listdir(SAIDA_DRIVE)), (
    'A pasta resultados/semana8/e5 já tem arquivos. Não sobrescrevo: me avise antes de rodar de novo.')

def sha256_arquivo(caminho):
    h = hashlib.sha256()
    with open(caminho, 'rb') as f:
        for bloco in iter(lambda: f.read(1 << 24), b''):
            h.update(bloco)
    return h.hexdigest()

CHAVE_RX = re.compile(r'([A-Za-z-]+_\d+)')
def chave_de(nome):
    achado = CHAVE_RX.search(os.path.basename(str(nome)))
    assert achado, f'Nome de chip sem chave reconhecível: {nome!r}'
    return achado.group(1)

REPO_URL = 'https://github.com/RoseBorges44/prithvi-confidence-shift'
REPO_GIT = '/content/repo'
if not os.path.isdir(REPO_GIT):
    subprocess.run(['git', 'clone', '-q', REPO_URL, REPO_GIT], check=True)
else:
    subprocess.run(['git', '-C', REPO_GIT, 'pull', '-q'], check=True)
commit = subprocess.run(['git', '-C', REPO_GIT, 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
tabela = open(f'{REPO_GIT}/analysis/README.md', encoding='utf-8').read()
registrados = dict(re.findall(r'\| `([a-z_]+\.py)` \| `([0-9a-f]{64})` \|', tabela))
assert set(registrados) == {'analise_congelada.py', 'nucleo.py', 'reamostragem.py', 'categorias.py'}, registrados.keys()
for nome, sha in registrados.items():
    assert sha256_arquivo(f'{REPO_GIT}/analysis/{nome}') == sha, f'{nome} é diferente do congelado. Não rode.'
data_plano = subprocess.run(['git', '-C', REPO_GIT, 'log', '--diff-filter=A', '-1', '--format=%cI %h', '--',
                             'prereg/plano_exploratorio.md'], capture_output=True, text=True, check=True).stdout.strip()
SHA_E5 = sha256_arquivo(f'{REPO_GIT}/analysis/exploratoria/e5_r0.py')
sys.path.insert(0, f'{REPO_GIT}/analysis/exploratoria')
sys.path.insert(0, f'{REPO_GIT}/analysis')
print(f'Repositório no commit {commit[:7]}. Scripts congelados conferem. Plano adicionado em {data_plano}.')

In [ ]:
from huggingface_hub import hf_hub_download

REPO = 'ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL'
ARQUIVO = 'Prithvi_EO_V2_300M_TL.pt'
REVISAO = '63adbd39c271da4c42f447e69b1a7c91a338cdc9'
SHA_BACKBONE = '3629cedfbb350faafcb0dac902ae0d3c927e25ce8d9e0024aa1276ec66956ddb'

no_drive = f'{RAIZ}/modelo_fixado/backbone/{ARQUIVO}'
if not os.path.exists(no_drive):
    print('Backbone ausente no Drive; baixando na revisão fixada.')
    hf_hub_download(REPO, ARQUIVO, revision=REVISAO, local_dir=os.path.dirname(no_drive))
BACKBONE_PT = f'/content/{ARQUIVO}'
if not (os.path.exists(BACKBONE_PT) and sha256_arquivo(BACKBONE_PT) == SHA_BACKBONE):
    shutil.copy(no_drive, BACKBONE_PT)
assert sha256_arquivo(BACKBONE_PT) == SHA_BACKBONE, 'sha256 do backbone diferente do registrado em modelo.md.'
print(f'Backbone conferido ({SHA_BACKBONE[:8]}…), cópia local em {BACKBONE_PT}')

In [ ]:
import yaml
SHA_PESOS_R0 = '3675e9c2b52547de8ff8a19f4881c28573e6d4d2f0805d866f4fc48c1e517d60'
SHA_CONFIG_R0 = 'ecc9ba411c7e2ca0be70bb597e0aae45d866acaf1fdfd2f4b8ac652ee5eab0b0'

PESOS_R0 = '/content/r0_pesos.pt'
if not (os.path.exists(PESOS_R0) and sha256_arquivo(PESOS_R0) == SHA_PESOS_R0):
    candidatos = [f'{FIXADO}/{f}' for f in sorted(os.listdir(FIXADO))
                  if f.endswith(('.pt', '.ckpt', '.pth')) and os.path.getsize(f'{FIXADO}/{f}') > 100e6]
    achado = next((p for p in candidatos if sha256_arquivo(p) == SHA_PESOS_R0), None)
    assert achado, 'Os pesos do R0 (SHA-256 3675e9c2…) não estão em Mestrado/modelo_fixado.'
    shutil.copy(achado, PESOS_R0)
assert sha256_arquivo(PESOS_R0) == SHA_PESOS_R0
CONFIG_R0 = f'{FIXADO}/config.yaml'
assert sha256_arquivo(CONFIG_R0) == SHA_CONFIG_R0, 'config.yaml do R0 diferente do registrado em modelo.md.'
print(f'Pesos e config.yaml do R0 conferidos (commit 918b9f1).')

cfg = yaml.safe_load(open(CONFIG_R0))
def procurar(no, chave):
    if isinstance(no, dict):
        if chave in no:
            return no[chave]
        for v in no.values():
            r = procurar(v, chave)
            if r is not None:
                return r
    elif isinstance(no, list):
        for v in no:
            r = procurar(v, chave)
            if r is not None:
                return r
    return None

RECEITA_S6 = dict(backbone='prithvi_eo_v2_300_tl', decoder='UperNetDecoder', decoder_channels=256,
                  decoder_scale_modules=True, head_dropout=0.1, num_classes=2, rescale=True,
                  necks=[{'name': 'SelectIndices', 'indices': [5, 11, 17, 23]}, {'name': 'ReshapeTokensToImage'}])
args_cfg = procurar(cfg, 'model_args') or {}
print('Receita do config.yaml do R0 contra a da semana 6:')
diferencas = []
for k, v in RECEITA_S6.items():
    no_cfg = args_cfg.get(k, '(ausente)')
    igual = no_cfg == v
    print(f'  {k:22s} config.yaml: {no_cfg!r:60.60s} {"igual" if igual else "DIFERENTE"}')
    if not igual:
        diferencas.append(k)
assert not {'backbone', 'decoder', 'num_classes'} & set(diferencas), 'Arquitetura do R0 diferente da montagem da semana 6.'
for chave in ('means', 'stds', 'bands', 'constant_scale', 'no_data_replace'):
    print(f'  dados, {chave:15s}: {procurar(cfg.get("data", cfg), chave)!r}')
print('As diferenças de dados acima (se houver) são justamente o que o E5 mede.')

In [ ]:
import urllib.request, concurrent.futures as cf

BUCKET = 'https://storage.googleapis.com/sen1floods11/v1.1/'
DADOS = '/content/sen1floods11'
S2 = f'{DADOS}/v1.1/data/flood_events/HandLabeled/S2Hand'
LB = f'{DADOS}/v1.1/data/flood_events/HandLabeled/LabelHand'
for d in (S2, LB):
    os.makedirs(d, exist_ok=True)
hashes = {l['arquivo']: l['sha256'] for l in csv.DictReader(open(HASHES_S1F11, newline=''))}
CHAVES_S1F11 = {conj: [str(c) for c in np.load(f'{MARGENS}/labels_{conj}.npz', allow_pickle=False)['chaves']]
                for conj in ('val', 'teste', 'bolivia')}
assert [len(v) for v in CHAVES_S1F11.values()] == [89, 90, 15]

def baixar(trabalho):
    url, destino = trabalho
    nome = os.path.basename(destino)
    if not (os.path.exists(destino) and sha256_arquivo(destino) == hashes[nome]):
        for tentativa in range(4):
            try:
                urllib.request.urlretrieve(url, destino + '.parcial')
                os.replace(destino + '.parcial', destino)
                break
            except Exception:
                time.sleep(2 * (tentativa + 1))
    assert sha256_arquivo(destino) == hashes[nome], f'{nome} diferente do hash gravado na semana 6.'
    return nome

trabalhos = []
for chave in [c for lista in CHAVES_S1F11.values() for c in lista]:
    trabalhos.append((f'{BUCKET}data/flood_events/HandLabeled/S2Hand/{chave}_S2Hand.tif', f'{S2}/{chave}_S2Hand.tif'))
    trabalhos.append((f'{BUCKET}data/flood_events/HandLabeled/LabelHand/{chave}_LabelHand.tif', f'{LB}/{chave}_LabelHand.tif'))
t0 = time.perf_counter()
with cf.ThreadPoolExecutor(16) as pool:
    feitos = list(pool.map(baixar, trabalhos))
print(f'{len(feitos)} arquivos conferidos pelos hashes da semana 6 em {time.perf_counter() - t0:.0f} s.')

In [ ]:
import rasterio
from scipy.ndimage import maximum_filter

TAM = 512
IDX_BANDAS = [1, 2, 3, 8, 11, 12]          # B2, B3, B4, B8A, B11, B12 nos 13 canais do Sen1Floods11
ESCALA = np.float32(0.0001)
RAIO_BORDA = 2

def borda_de(y):
    """Verdadeiro nos pixels de água ou terra a até 2 pixels (janela 5 x 5) de um pixel da outra classe.
    Pixels com rótulo -1 não formam borda."""
    agua = (y == 1).astype(np.uint8)
    terra = (y == 0).astype(np.uint8)
    lado = 2 * RAIO_BORDA + 1
    perto_agua = maximum_filter(agua, size=lado, mode='constant', cval=0).astype(bool)
    perto_terra = maximum_filter(terra, size=lado, mode='constant', cval=0).astype(bool)
    return ((agua == 1) & perto_terra) | ((terra == 1) & perto_agua)

PASTA_CACHE = '/content/cache_imagens'

def alocar(nome, forma, tipo):
    """Matriz de imagens no disco local (memmap), para não ocupar a RAM da sessão."""
    os.makedirs(PASTA_CACHE, exist_ok=True)
    return np.lib.format.open_memmap(f'{PASTA_CACHE}/{nome}.npy', mode='w+', dtype=tipo, shape=forma)

def preencher(x, valor):
    x = np.ma.filled(x.astype(np.float32), np.nan)
    return np.where(np.isnan(x), np.float32(valor), x).astype(np.float32)

def ler_chip(chave, pasta_s2, pasta_lb):
    with rasterio.open(f'{pasta_s2}/{chave}_S2Hand.tif') as s:
        assert s.count == 13 and (s.height, s.width) == (TAM, TAM), f'{chave}: imagem {s.count} x {s.height} x {s.width}'
        x = preencher(s.read(masked=True), 0.0)[IDX_BANDAS]
    with rasterio.open(f'{pasta_lb}/{chave}_LabelHand.tif') as s:
        assert s.count == 1 and (s.height, s.width) == (TAM, TAM), f'{chave}: rótulo {s.count} x {s.height} x {s.width}'
        y = preencher(s.read(1, masked=True), -1.0).astype(np.int16)
    assert set(np.unique(y).tolist()) <= {-1, 0, 1}, f'{chave}: valores de rótulo {np.unique(y)}'
    return x, y

def montar_s1f11(chaves, pasta_s2, pasta_lb, nome='s1f11'):
    n = len(chaves)
    bruto = alocar(nome, (n, len(IDX_BANDAS), TAM, TAM), np.float32)
    ys = np.empty((n, TAM, TAM), np.int16)
    bordas = np.empty((n, TAM, TAM), bool)
    for i, c in enumerate(chaves):
        bruto[i], ys[i] = ler_chip(c, pasta_s2, pasta_lb)
        bordas[i] = borda_de(ys[i])
    bruto.flush()
    return dict(chaves=list(chaves), bruto=bruto, ys=ys, bordas=bordas)

def montar_t2(linhas, pasta_janelas, pasta_wf2):
    """linhas: linhas das tabelas de janelas que entram na sensibilidade, em qualquer ordem."""
    linhas = sorted(linhas, key=lambda r: r['window_id'])
    n = len(linhas)
    bruto = alocar('t2', (n, len(IDX_BANDAS), TAM, TAM), np.int16)
    ys = np.empty((n, TAM, TAM), np.int16)
    bordas = np.empty((n, TAM, TAM), bool)
    posicao = {r['window_id']: i for i, r in enumerate(linhas)}
    assert len(posicao) == n, 'Janelas repetidas.'
    for mapa in sorted({r['map_id'] for r in linhas}):
        with rasterio.open(f'{pasta_wf2}/test/gt/{mapa}.tif') as g:
            gt = g.read()
        cena = np.full(gt.shape[1:], -1, np.int16)
        cena[gt[1] == 1] = 0
        cena[gt[1] == 2] = 1
        cena[gt[0] != 1] = -1
        borda_cena = borda_de(cena)
        for r in (r for r in linhas if r['map_id'] == mapa):
            i, jid = posicao[r['window_id']], r['window_id']
            lin, col = int(r['row_off']), int(r['col_off'])
            with rasterio.open(f'{pasta_janelas}/S2/{jid}.tif') as s:
                assert s.count == 6 and (s.height, s.width) == (TAM, TAM) and s.dtypes[0] == 'int16', f'{jid}: imagem'
                bruto[i] = s.read()
            with rasterio.open(f'{pasta_janelas}/label/{jid}.tif') as s:
                assert s.count == 1 and (s.height, s.width) == (TAM, TAM), f'{jid}: rótulo'
                y = preencher(s.read(1, masked=True), -1.0).astype(np.int16)
            ys[i] = y
            recorte = cena[lin:lin + TAM, col:col + TAM]
            pont = y >= 0
            assert np.array_equal(y[pont], recorte[pont]), f'{jid}: rótulo da janela diferente do recorte da cena'
            bordas[i] = borda_cena[lin:lin + TAM, col:col + TAM]
            n_pont, n_agua = int(pont.sum()), int((y == 1).sum())
            assert (n_pont, n_agua, n_pont - n_agua) == (int(r['scored_pixels']), int(r['water_pixels']), int(r['land_pixels'])), (
                f'{jid}: contagens diferentes das da tabela de janelas')
    bruto.flush()
    return dict(chaves=[r['window_id'] for r in linhas], bruto=bruto, ys=ys, bordas=bordas)

def rotulos_do_conjunto(c, com_borda):
    pont = c['ys'] >= 0
    saida = dict(chaves=np.array(c['chaves']), tamanhos=pont.reshape(len(c['chaves']), -1).sum(1).astype(np.int64),
                 rotulo=c['ys'][pont].astype(np.uint8))
    if com_borda:
        saida['borda'] = c['bordas'][pont]
    return saida

def gravar(caminho, **campos):
    tmp = os.path.join(os.path.dirname(caminho), 'parcial_' + os.path.basename(caminho))
    np.savez(tmp, **campos)
    os.replace(tmp, caminho)

In [ ]:
t0 = time.perf_counter()
CONJ = {conj: montar_s1f11(CHAVES_S1F11[conj], S2, LB, nome=conj) for conj in ('val', 'teste', 'bolivia')}
for conj, c in CONJ.items():
    r = rotulos_do_conjunto(c, com_borda=(conj != 'val'))
    antigo = np.load(f'{MARGENS}/labels_{conj}.npz', allow_pickle=False)
    assert set(antigo.files) == set(r) and all(np.array_equal(antigo[k], r[k]) for k in r), (
        f'{conj}: rótulos diferentes dos da semana 6.')
    c.pop('bordas')
    print(f'{conj}: {len(c["chaves"])} chips, rótulos iguais aos da semana 6 ({time.perf_counter() - t0:.0f} s)')

In [ ]:
import warnings, contextlib, io, logging, gc
import torch
import albumentations as A
from albumentations.pytorch import ToTensorV2
from importlib.metadata import version
from scipy.special import log_expit, logsumexp
try:
    from terratorch.tasks import SemanticSegmentationTask
    from terratorch.datamodules import Sen1Floods11NonGeoDataModule
except ModuleNotFoundError as erro:
    raise RuntimeError('TerraTorch não está instalado nesta sessão. Rode a célula 1 e confira que ela imprime terratorch 1.2.11.') from erro

warnings.filterwarnings('ignore', category=UserWarning)
assert torch.cuda.is_available(), 'Sem GPU: Ambiente de execução > Alterar tipo de ambiente de execução > T4.'
DISPOSITIVO = 'cuda'
BANDAS = ['BLUE', 'GREEN', 'RED', 'NIR_NARROW', 'SWIR_1', 'SWIR_2']
ESPERADO_M = {'R1': 15.1, 'R2': 20.6, 'R3': 319.0}
TOLERANCIA_M = 0.1
LR = {'R1': 5e-4, 'R2': 5e-4, 'R3': 5e-5}
EPOCAS = 50
LOTE_INF = 8
LIMITE_DIF_VAL = 0.05

torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

def datamodule():
    para_tensor = lambda: ToTensorV2(transpose_mask=False)
    return Sen1Floods11NonGeoDataModule(
        data_root=DADOS, batch_size=LOTE_INF, num_workers=0, bands=BANDAS,
        train_transform=[para_tensor()], val_transform=[para_tensor()], test_transform=[para_tensor()],
        drop_last=False, constant_scale=0.0001, no_data_replace=0.0, no_label_replace=-1, use_metadata=False)

LORA = dict(method='LORA', replace_qkv='qkv',
            peft_config_kwargs=dict(target_modules=['qkv.q_linear', 'qkv.v_linear', 'mlp.fc1', 'mlp.fc2'],
                                    lora_alpha=16, r=16))

def tarefa(regime):
    args = dict(
        backbone='prithvi_eo_v2_300_tl', backbone_pretrained=True, backbone_ckpt_path=BACKBONE_PT,
        backbone_bands=BANDAS,
        necks=[{'name': 'SelectIndices', 'indices': [5, 11, 17, 23]}, {'name': 'ReshapeTokensToImage'}],
        decoder='UperNetDecoder', decoder_channels=256, decoder_scale_modules=True,
        head_dropout=0.1, num_classes=2, rescale=True)
    if regime == 'R2':
        args['peft_config'] = LORA
    t = SemanticSegmentationTask(
        model_factory='EncoderDecoderFactory', model_args=args, loss='ce', ignore_index=-1,
        lr=LR[regime], optimizer='AdamW', optimizer_hparams={'weight_decay': 0.05},
        scheduler='CosineAnnealingLR', scheduler_hparams={'T_max': EPOCAS, 'eta_min': 0},
        freeze_backbone=(regime == 'R1'), plot_on_val=0)
    t.configure_models()
    treinaveis = sum(p.numel() for p in t.parameters() if p.requires_grad)
    if regime == 'R2':
        n_lora = sum(1 for n, _ in t.model.encoder.named_parameters() if 'lora_A' in n)
        assert n_lora == 96, 'Número de módulos LoRA diferente do esperado.'
    assert abs(treinaveis / 1e6 - ESPERADO_M[regime]) <= TOLERANCIA_M, 'Parâmetros treináveis fora do esperado.'
    return t

DM = datamodule()

@contextlib.contextmanager
def em_silencio():
    """Esconde o que a construção do modelo imprime, que diferiria entre os regimes."""
    logging.disable(logging.CRITICAL)
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            yield
    finally:
        logging.disable(logging.NOTSET)

def carregar_modelo(regime, semente):
    with em_silencio():
        t = tarefa(regime)
        t.model.to(DISPOSITIVO)
        gc.collect()
        pesos = torch.load(PESOS[(regime, semente)]['pesos'], map_location=DISPOSITIVO)
        do_modelo = set(t.model.state_dict().keys())
        esperado = do_modelo if regime != 'R1' else {k for k in do_modelo if not k.startswith('encoder.')}
        assert set(pesos.keys()) == esperado, 'Os tensores salvos não coincidem com os do modelo.'
        resultado = t.model.load_state_dict(pesos, strict=False)
        assert not resultado.unexpected_keys, 'Tensores salvos sem lugar no modelo.'
        assert set(resultado.missing_keys) == (do_modelo - esperado), 'Tensores do modelo sem peso.'
        modelo = t.model.to(DISPOSITIVO).float().eval()
        for p in modelo.parameters():
            p.requires_grad_(False)
        del t, pesos
    gc.collect()
    return modelo

TRANSFORMACOES = [(k, espelho) for espelho in (False, True) for k in range(4)]

def aplicar(x, k, espelho):
    if espelho:
        x = torch.flip(x, dims=(-1,))
    return torch.rot90(x, k, dims=(-2, -1))

def desfazer(y, k, espelho):
    y = torch.rot90(y, -k, dims=(-2, -1))
    if espelho:
        y = torch.flip(y, dims=(-1,))
    return y

def margem_media_probabilidade(pilha):
    """Logit da probabilidade média de água de K margens, pilha (K, N), em float64 (igual a nucleo.mean_probability_margin)."""
    pilha = np.asarray(pilha, dtype=np.float64)
    k = pilha.shape[0]
    lp = logsumexp(log_expit(pilha), axis=0) - np.log(k)
    lq = logsumexp(log_expit(-pilha), axis=0) - np.log(k)
    return lp - lq

def inferir(modelo, conj, tta=False):
    """Margem de logit (água menos terra) de cada pixel pontuado, em 32 bits, chips inteiros.
    Com tta=True, logit da probabilidade média sobre as 8 transformações do grupo diedral."""
    bruto, ys = conj['bruto'], conj['ys']
    margens = []
    with torch.no_grad(), torch.autocast(device_type=DISPOSITIVO, enabled=False):
        for i in range(0, len(ys), LOTE_INF):
            x = torch.from_numpy(bruto[i:i + LOTE_INF].astype(np.float32) * ESCALA).to(DISPOSITIVO)
            y = torch.from_numpy(ys[i:i + LOTE_INF].astype(np.int64)).to(DISPOSITIVO)
            x = DM.aug({'image': x, 'mask': y})['image']
            assert x.dtype == torch.float32 and x.ndim == 4 and x.shape[1] == len(BANDAS), tuple(x.shape)
            pilhas = []
            for k, espelho in (TRANSFORMACOES if tta else [(0, False)]):
                z = modelo(aplicar(x, k, espelho)).output.float()
                assert tuple(z.shape[-2:]) == tuple(x.shape[-2:]), f'Saída {tuple(z.shape)}'
                pilhas.append(desfazer(z[:, 1] - z[:, 0], k, espelho).cpu().numpy())
            for j in range(x.shape[0]):
                pont = ys[i + j] >= 0
                if tta:
                    margens.append(margem_media_probabilidade(np.stack([p[j][pont] for p in pilhas])).astype(np.float32))
                else:
                    margens.append(pilhas[0][j][pont].astype(np.float32))
    return np.concatenate(margens)

def comparar_val(regime, semente, chaves, tamanhos, margem):
    z = np.load(PESOS[(regime, semente)]['margens_val'], allow_pickle=False)
    ch_o, tam_o, marg_o = [str(c) for c in z['chaves']], z['tamanhos'], z['margem']
    pos = {c: i for i, c in enumerate(ch_o)}
    off_o = np.concatenate([[0], np.cumsum(tam_o)])
    off_n = np.concatenate([[0], np.cumsum(tamanhos)])
    dif_max, trocas = 0.0, 0
    for i, c in enumerate(chaves):
        j = pos[str(c)]
        assert int(tam_o[j]) == int(tamanhos[i])
        a, b = marg_o[off_o[j]:off_o[j + 1]], margem[off_n[i]:off_n[i + 1]]
        if a.size:
            dif_max = max(dif_max, float(np.max(np.abs(a - b))))
            trocas += int(np.sum((a > 0) != (b > 0)))
    return dif_max, trocas

print('Definições carregadas. Nada foi inferido ainda.')

In [ ]:
from e5_r0 import normalizar_chaves

def carregar_r0():
    t = tarefa('R3')                     # montagem completa da semana 6, nada congelado
    try:
        ck = torch.load(PESOS_R0, map_location='cpu', weights_only=True)
    except Exception:
        ck = torch.load(PESOS_R0, map_location='cpu', weights_only=False)   # arquivo conferido pelo SHA-256
    estado = ck.get('state_dict', ck) if isinstance(ck, dict) else ck
    do_modelo = list(t.model.state_dict().keys())
    mapa, sobras = normalizar_chaves(estado.keys(), do_modelo)
    t.model.load_state_dict({k: estado[mapa[k]] for k in do_modelo}, strict=True)
    print(f'R0 carregado: {len(do_modelo)} tensores do modelo preenchidos pelo checkpoint; '
          f'{len(sobras)} tensores do checkpoint fora do modelo{": " + ", ".join(sobras[:5]) if sobras else ""}.')
    modelo = t.model.to(DISPOSITIVO).float().eval()
    for p in modelo.parameters():
        p.requires_grad_(False)
    del t, ck, estado
    gc.collect()
    return modelo

MODELO_R0 = carregar_r0()

In [ ]:
MARG_R0 = '/content/margens_r0'
os.makedirs(MARG_R0, exist_ok=True)
for conj in ('val', 'teste', 'bolivia'):
    t0 = time.perf_counter()
    c = CONJ[conj]
    m = inferir(MODELO_R0, c)
    pont = c['ys'] >= 0
    tamanhos = pont.reshape(len(c['chaves']), -1).sum(1).astype(np.int64)
    assert m.dtype == np.float32 and m.size == int(tamanhos.sum()) and np.isfinite(m).all()
    np.savez(f'{MARG_R0}/r0_{conj}.npz', chaves=np.array(c['chaves']), tamanhos=tamanhos, margem=m)
    print(f'{conj}: {m.size:,} pixels inferidos em {time.perf_counter() - t0:.0f} s')

In [ ]:
SAIDA = '/content/e5'
LOG = '/content/e5.log'
if os.path.isdir(SAIDA):
    shutil.rmtree(SAIDA)
cmd = (f'cd {REPO_GIT} && set -o pipefail && python analysis/exploratoria/e5_r0.py --margens-r0 {MARG_R0} '
       f'--margens "{MARGENS}" --confirmatoria "{CONFIRMATORIA}" --saida {SAIDA} 2>&1 | tee {LOG}')
get_ipython().system(f"bash -c '{cmd}'")
assert os.path.isfile(f'{SAIDA}/e5.json'), 'O cálculo não terminou. Guarde a saída e me mande.'
os.makedirs(SAIDA_DRIVE, exist_ok=True)
shutil.copytree(SAIDA, SAIDA_DRIVE, dirs_exist_ok=True)
shutil.copytree(MARG_R0, f'{SAIDA_DRIVE}/margens_r0', dirs_exist_ok=True)
shutil.copy(LOG, f'{SAIDA_DRIVE}/e5.log')
from importlib.metadata import version
INFO = {'commit_repositorio': commit, 'plano': 'prereg/plano_exploratorio.md, E5', 'plano_adicionado': data_plano,
        'e5_sha256': SHA_E5, 'congelados_sha256': registrados, 'pesos_r0_sha256': SHA_PESOS_R0,
        'config_r0_sha256': SHA_CONFIG_R0, 'receita_diferente_em': diferencas,
        'versoes': {p: version(p) for p in ('terratorch', 'torchgeo', 'torch', 'numpy')},
        'gpu': torch.cuda.get_device_name(0), 'gerado_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}
json.dump(INFO, open(f'{SAIDA_DRIVE}/rodada.json', 'w'), indent=2)
print(open(f'{SAIDA}/e5_r0.csv').read())
print(f'Saída copiada para {SAIDA_DRIVE}.')